# AIONOS Assignment 1 — Executive Productivity Agent

**Role:** Arjun Malhotra, VP Sales  
**Purpose:** Convert supplied emails, calendar events, meeting notes, and voice notes into a clear executive action brief.

Prototype features: **My Actions, Waiting on Others, Unclear Ownership, deadlines/status, Daily Action Brief, Q&A, and source grounding.**

## 1. Workflow

**Input data → Extract commitments → Classify → Check status/deadlines → Generate Daily Brief → Answer questions with source evidence**

In [1]:
import json
from pathlib import Path
print('Python environment ready.')

Python environment ready.


## 2. Load the assignment data

If `data.json` is in the same folder, it is loaded. Otherwise, the notebook uses the structured assignment information below.

In [4]:
data_file = Path('data.json')
if data_file.exists():
    with open(data_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    print('data.json loaded successfully.')
else:
    data = {
        'executive': {'name':'Arjun Malhotra','role':'VP Sales','email':'arjun.malhotra@veridian-corp.example'},
        'commitments': [
            {'id':'C1','title':'Send updated vendor list to Raghav','category':'my_action','status':'pending','priority':'high','deadline':'Wednesday 23 Sep morning','owner':'Arjun Malhotra','sources':['Leadership Sync','Thread 1 Vendor List','Voice Note 1']},
            {'id':'C2','title':'Review Q3 campaign deck','category':'my_action','status':'ready','priority':'medium','deadline':'Thursday 24 Sep, 9:30 AM','owner':'Arjun Malhotra','sources':['Leadership Sync','Thread 2 Q3 Campaign Deck']},
            {'id':'C3','title':'Review July expense variance report','category':'my_action','status':'received_needs_review','priority':'medium','deadline':'Before Thursday 24 Sep board prep','owner':'Arjun Malhotra','sources':['Leadership Sync','Thread 4 Expense Variance','Voice Note 2']},
            {'id':'C4','title':'Attend Meridian Logistics client call','category':'my_action','status':'confirmed','priority':'medium','deadline':'Wednesday 23 Sep, 3:00 PM','owner':'Arjun Malhotra','sources':['Leadership Sync','Thread 3 Call Reschedule']},
            {'id':'C5','title':'Resolve ownership / authorized sign-off for Mumbai office lease renewal','category':'unclear_ownership','status':'unassigned','priority':'critical','deadline':'Friday 25 Sep, 5:00 PM','owner':'Unassigned','sources':['Leadership Sync','Thread 5 Mumbai Office Lease']}
        ],
        'waiting_on_others': [
            {'id':'W1','title':'Q3 campaign deck from Neha','status':'completed_by_source','person':'Neha Kapoor','sources':['Thread 2 Q3 Campaign Deck']},
            {'id':'W2','title':'July expense variance report from Divya','status':'completed_by_source','person':'Divya Rao','sources':['Thread 4 Expense Variance']},
            {'id':'W3','title':'Meridian Logistics call confirmation from Priya','status':'confirmed','person':'Priya Nair','sources':['Thread 3 Call Reschedule']}
        ]
    }
    print('data.json not found; using the supplied assignment data for the demo.')
print('Executive:', data['executive']['name'], '|', data['executive']['role'])

data.json not found; using the supplied assignment data for the demo.
Executive: Arjun Malhotra | VP Sales


## 3. My Actions

In [7]:
my_actions = [x for x in data['commitments'] if x['category'] == 'my_action']
for x in my_actions:
    print(f"{x['id']} | {x['title']} | {x['status']} | {x['priority']} | {x['deadline']}")

C1 | Send updated vendor list to Raghav | pending | high | Wednesday 23 Sep morning
C2 | Review Q3 campaign deck | ready | medium | Thursday 24 Sep, 9:30 AM
C3 | Review July expense variance report | received_needs_review | medium | Before Thursday 24 Sep board prep
C4 | Attend Meridian Logistics client call | confirmed | medium | Wednesday 23 Sep, 3:00 PM


## 4. Waiting on Others

In [10]:
waiting_items = data['waiting_on_others']
for x in waiting_items:
    print(f"{x['id']} | {x['title']} | {x['person']} | {x['status']}")

W1 | Q3 campaign deck from Neha | Neha Kapoor | completed_by_source
W2 | July expense variance report from Divya | Divya Rao | completed_by_source
W3 | Meridian Logistics call confirmation from Priya | Priya Nair | confirmed


## 5. Unclear Ownership

In [13]:
unclear_items = [x for x in data['commitments'] if x['category'] == 'unclear_ownership']
for x in unclear_items:
    print(f"{x['id']} | {x['title']} | Owner: {x['owner']} | Deadline: {x['deadline']} | Priority: {x['priority']}")
print('Rule: do not guess the owner when the source does not confirm it.')

C5 | Resolve ownership / authorized sign-off for Mumbai office lease renewal | Owner: Unassigned | Deadline: Friday 25 Sep, 5:00 PM | Priority: critical
Rule: do not guess the owner when the source does not confirm it.


## 6. Daily Action Brief

In [16]:
priority_order = {'critical':0,'high':1,'medium':2,'low':3}
sorted_actions = sorted(my_actions, key=lambda x: priority_order.get(x['priority'], 9))
print('='*72)
print('DAILY ACTION BRIEF — ARJUN MALHOTRA')
print('='*72)
print('\nACTIONS TO HANDLE:')
for x in sorted_actions:
    print(f"• {x['title']} — {x['status']} — {x['priority'].upper()} — {x['deadline']}")
print('\nUNCLEAR OWNERSHIP:')
for x in unclear_items:
    print(f"• {x['title']} — currently unassigned; confirm authorized owner/signatory.")
print('\nDEPENDENCIES:')
for x in waiting_items:
    print(f"• {x['title']} — {x['person']} — {x['status']}")
print('\nKEY CONFIRMED EVENT: Meridian Logistics client call — Wed 23 Sep, 3:00 PM')

DAILY ACTION BRIEF — ARJUN MALHOTRA

ACTIONS TO HANDLE:
• Send updated vendor list to Raghav — pending — HIGH — Wednesday 23 Sep morning
• Review Q3 campaign deck — ready — MEDIUM — Thursday 24 Sep, 9:30 AM
• Review July expense variance report — received_needs_review — MEDIUM — Before Thursday 24 Sep board prep
• Attend Meridian Logistics client call — confirmed — MEDIUM — Wednesday 23 Sep, 3:00 PM

UNCLEAR OWNERSHIP:
• Resolve ownership / authorized sign-off for Mumbai office lease renewal — currently unassigned; confirm authorized owner/signatory.

DEPENDENCIES:
• Q3 campaign deck from Neha — Neha Kapoor — completed_by_source
• July expense variance report from Divya — Divya Rao — completed_by_source
• Meridian Logistics call confirmation from Priya — Priya Nair — confirmed

KEY CONFIRMED EVENT: Meridian Logistics client call — Wed 23 Sep, 3:00 PM


## 7. Q&A Agent

In [19]:
def ask_agent(question):
    q = question.lower()
    if 'promise' in q and 'raghav' in q:
        x = next(i for i in data['commitments'] if i['id']=='C1')
        return f"You promised Raghav the updated vendor list. Status: {x['status']}. Deadline: {x['deadline']}. Sources: {', '.join(x['sources'])}."
    if 'action' in q or 'todo' in q or 'to do' in q:
        return '\n'.join(['Actions requiring attention:'] + [f"- {x['title']} ({x['status']})" for x in sorted_actions] + ['- Confirm ownership of the Mumbai office lease renewal.'])
    if 'mumbai' in q or 'lease' in q:
        x = next(i for i in data['commitments'] if i['id']=='C5')
        return f"The Mumbai office lease renewal is still unassigned. Deadline: {x['deadline']}. The supplied data does not confirm who should sign, so ownership should be confirmed rather than assumed. Sources: {', '.join(x['sources'])}."
    if 'campaign' in q or 'deck' in q:
        return 'The Q3 campaign deck is ready for review. Review: Thursday 24 Sep at 9:30 AM. Source: Thread 2 Q3 Campaign Deck.'
    if 'expense' in q or 'variance' in q:
        return 'Divya sent the July expense variance report Wednesday evening. Arjun needs to review it before Thursday board prep. Source: Thread 4 Expense Variance.'
    if 'meridian' in q or 'client call' in q:
        return 'The Meridian Logistics client call is confirmed for Wednesday 23 Sep at 3:00 PM. Source: Thread 3 Call Reschedule.'
    return 'I could not find a directly supported answer in the supplied data.'

## 8. Test the Agent

In [22]:
questions = ['What did I promise Raghav?','What needs action today?','What is the status of the Mumbai lease?','What is happening with the campaign deck?','What about the expense variance report?','When is the Meridian client call?']
for q in questions:
    print('\nUSER:', q)
    print('AGENT:', ask_agent(q))


USER: What did I promise Raghav?
AGENT: You promised Raghav the updated vendor list. Status: pending. Deadline: Wednesday 23 Sep morning. Sources: Leadership Sync, Thread 1 Vendor List, Voice Note 1.

USER: What needs action today?
AGENT: Actions requiring attention:
- Send updated vendor list to Raghav (pending)
- Review Q3 campaign deck (ready)
- Review July expense variance report (received_needs_review)
- Attend Meridian Logistics client call (confirmed)
- Confirm ownership of the Mumbai office lease renewal.

USER: What is the status of the Mumbai lease?
AGENT: The Mumbai office lease renewal is still unassigned. Deadline: Friday 25 Sep, 5:00 PM. The supplied data does not confirm who should sign, so ownership should be confirmed rather than assumed. Sources: Leadership Sync, Thread 5 Mumbai Office Lease.

USER: What is happening with the campaign deck?
AGENT: The Q3 campaign deck is ready for review. Review: Thursday 24 Sep at 9:30 AM. Source: Thread 2 Q3 Campaign Deck.

USER: W

## 9. Source Grounding

In [25]:
for x in data['commitments']:
    print(f"\n{x['id']} — {x['title']}")
    print('Sources:', ', '.join(x['sources']))


C1 — Send updated vendor list to Raghav
Sources: Leadership Sync, Thread 1 Vendor List, Voice Note 1

C2 — Review Q3 campaign deck
Sources: Leadership Sync, Thread 2 Q3 Campaign Deck

C3 — Review July expense variance report
Sources: Leadership Sync, Thread 4 Expense Variance, Voice Note 2

C4 — Attend Meridian Logistics client call
Sources: Leadership Sync, Thread 3 Call Reschedule

C5 — Resolve ownership / authorized sign-off for Mumbai office lease renewal
Sources: Leadership Sync, Thread 5 Mumbai Office Lease


## 10. Final Result

This notebook demonstrates the required prototype logic: extracting executive commitments, separating actions/dependencies, tracking status and deadlines, flagging unclear ownership, generating a Daily Action Brief, and answering questions using source references.

